# KMeans Clustering Analysis

This notebook performs unsupervised clustering using KMeans algorithm.

**Analysis includes:**
- Testing K values in [2, 3, 4, 5, 6]
- Computing silhouette scores and SSE (inertia) for each K
- Selecting best K using highest silhouette score
- Visualizing clusters using PCA (2 components)
- Providing interpretation of cluster separability

**Note**: Class labels are ignored for unsupervised learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
RANDOM_STATE = 42
sns.set_style('whitegrid')

print("✓ All libraries imported successfully!")

In [ ]:
# Record library versions for reproducibility
import sys
print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Random State: {RANDOM_STATE}")
print("="*60)

## 1. Load Data and Extract Features

Loading the dataset and ignoring class labels for unsupervised clustering.

In [ ]:
# Load dataset
df = pd.read_csv('my_data .csv')

# Extract features only (ignore class labels)
X = df.drop(columns=['placed'])

print(f"Dataset shape: {df.shape}")
print(f"Features shape (ignoring labels): {X.shape}")
print(f"\nFeature columns: {list(X.columns)}")
print(f"\nFeature types:")
print(X.dtypes)

## 2. Handle Categorical Features

Converting categorical features to numerical using one-hot encoding.

In [ ]:
# Identify categorical and numerical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

# One-hot encode categorical features if they exist
if len(cat_features) > 0:
    X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=False)
    print(f"\n✓ Categorical features encoded")
else:
    X_encoded = X.copy()
    print(f"\n✓ No categorical features to encode")

print(f"\nFinal feature matrix shape: {X_encoded.shape}")
print(f"Total features after encoding: {X_encoded.shape[1]}")

## 3. Scale Features with StandardScaler

Fitting StandardScaler on the full feature matrix X.

In [ ]:
# Initialize and fit StandardScaler on full X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print(f"✓ Features scaled using StandardScaler")
print(f"Scaled data shape: {X_scaled.shape}")
print(f"\nScaled data statistics:")
print(f"   Mean: {X_scaled.mean():.6f} (should be ~0)")
print(f"   Std:  {X_scaled.std():.6f} (should be ~1)")
print(f"   Min:  {X_scaled.min():.4f}")
print(f"   Max:  {X_scaled.max():.4f}")

## 4. Run KMeans for K in [2, 3, 4, 5, 6]

Testing different values of K and recording silhouette scores and SSE (inertia).

In [ ]:
# Define K values to test
k_values = [2, 3, 4, 5, 6]

# Store results
results = []
kmeans_models = {}

print("Running KMeans for different K values...")
print("="*70)

for k in k_values:
    # Train KMeans
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=RANDOM_STATE)
    cluster_labels = kmeans.fit_predict(X_scaled)
    
    # Calculate metrics
    silhouette = silhouette_score(X_scaled, cluster_labels)
    sse = kmeans.inertia_
    
    # Store results
    results.append({
        'K': k,
        'Silhouette Score': silhouette,
        'SSE (Inertia)': sse
    })
    
    # Store model
    kmeans_models[k] = kmeans
    
    print(f"K={k}: Silhouette={silhouette:.4f}, SSE={sse:.2f}")

print("="*70)
print("✓ KMeans clustering completed for all K values")

## 5. Create Scores Table

In [ ]:
# Create DataFrame with results
scores_df = pd.DataFrame(results)

print("\n" + "="*70)
print("KMEANS CLUSTERING SCORES")
print("="*70)
print(scores_df.to_string(index=False))
print("="*70)

# Find best K based on highest silhouette score
best_k_idx = scores_df['Silhouette Score'].idxmax()
best_k = scores_df.loc[best_k_idx, 'K']
best_silhouette = scores_df.loc[best_k_idx, 'Silhouette Score']
best_sse = scores_df.loc[best_k_idx, 'SSE (Inertia)']

print(f"\n🏆 BEST K: {best_k}")
print(f"   Silhouette Score: {best_silhouette:.4f}")
print(f"   SSE (Inertia):    {best_sse:.2f}")

## 6. Visualize Silhouette Scores and Elbow Curve

In [ ]:
# Create visualization for scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Silhouette Scores
axes[0].plot(scores_df['K'], scores_df['Silhouette Score'], 
             marker='o', linewidth=2, markersize=10, color='#2E86AB')
axes[0].axvline(x=best_k, color='red', linestyle='--', linewidth=2, alpha=0.7, 
                label=f'Best K={best_k}')
axes[0].scatter([best_k], [best_silhouette], color='red', s=200, zorder=5, 
                edgecolors='black', linewidth=2)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[0].set_title('Silhouette Score vs K', fontsize=14, fontweight='bold', pad=15)
axes[0].grid(alpha=0.3, linestyle='--')
axes[0].legend(fontsize=11)
axes[0].set_xticks(k_values)

# Add value labels
for k, score in zip(scores_df['K'], scores_df['Silhouette Score']):
    axes[0].annotate(f'{score:.3f}', xy=(k, score), 
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=9, fontweight='bold')

# Plot 2: Elbow Curve (SSE)
axes[1].plot(scores_df['K'], scores_df['SSE (Inertia)'], 
             marker='s', linewidth=2, markersize=10, color='#A23B72')
axes[1].axvline(x=best_k, color='red', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Best K={best_k}')
axes[1].scatter([best_k], [best_sse], color='red', s=200, zorder=5,
                edgecolors='black', linewidth=2)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('SSE (Inertia)', fontsize=12, fontweight='bold')
axes[1].set_title('Elbow Curve: SSE vs K', fontsize=14, fontweight='bold', pad=15)
axes[1].grid(alpha=0.3, linestyle='--')
axes[1].legend(fontsize=11)
axes[1].set_xticks(k_values)

# Add value labels
for k, sse in zip(scores_df['K'], scores_df['SSE (Inertia)']):
    axes[1].annotate(f'{sse:.0f}', xy=(k, sse),
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('kmeans_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Scores visualization saved as 'kmeans_scores.png'")

## 7. PCA Visualization of Best Clustering

Reducing dimensions to 2D using PCA and visualizing clusters.

In [ ]:
# Apply PCA for visualization (2 components)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Get cluster labels for best K
best_kmeans = kmeans_models[best_k]
cluster_labels = best_kmeans.predict(X_scaled)

# Get cluster centers in PCA space
cluster_centers_scaled = best_kmeans.cluster_centers_
cluster_centers_pca = pca.transform(cluster_centers_scaled)

print(f"✓ PCA applied: {X_scaled.shape[1]} features → 2 components")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"   Cluster {cluster}: {count} samples ({count/len(cluster_labels)*100:.1f}%)")

## 8. Create PCA Scatter Plot

In [ ]:
# Create PCA scatter plot
plt.figure(figsize=(12, 9))

# Define color palette
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']
cluster_colors = [colors[i % len(colors)] for i in range(best_k)]

# Plot each cluster
for i in range(best_k):
    cluster_mask = cluster_labels == i
    plt.scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1],
                c=cluster_colors[i], label=f'Cluster {i}',
                alpha=0.6, s=80, edgecolors='black', linewidth=0.5)

# Plot cluster centers
plt.scatter(cluster_centers_pca[:, 0], cluster_centers_pca[:, 1],
            c='red', marker='X', s=400, edgecolors='black', linewidth=2,
            label='Centroids', zorder=10)

# Add labels to centroids
for i, center in enumerate(cluster_centers_pca):
    plt.annotate(f'C{i}', xy=center, xytext=(5, 5), 
                 textcoords='offset points', fontsize=12, 
                 fontweight='bold', color='darkred')

plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.1%} variance)', 
           fontsize=13, fontweight='bold')
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.1%} variance)', 
           fontsize=13, fontweight='bold')
plt.title(f'KMeans Clustering Visualization (K={best_k})\nPCA 2D Projection', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(loc='best', fontsize=11, framealpha=0.9)
plt.grid(alpha=0.3, linestyle='--')

# Add text box with metrics
textstr = f'Silhouette Score: {best_silhouette:.4f}\nSSE: {best_sse:.2f}\nVariance Explained: {pca.explained_variance_ratio_.sum():.1%}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('kmeans_pca_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ PCA cluster visualization saved as 'kmeans_pca_clusters.png'")

## 9. Cluster Separability Analysis

In [ ]:
# Calculate additional metrics for cluster quality
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

calinski = calinski_harabasz_score(X_scaled, cluster_labels)
davies_bouldin = davies_bouldin_score(X_scaled, cluster_labels)

print("\n" + "="*70)
print("CLUSTER QUALITY METRICS")
print("="*70)
print(f"Silhouette Score:        {best_silhouette:.4f}  (higher is better, range: -1 to 1)")
print(f"Calinski-Harabasz Score: {calinski:.2f}  (higher is better)")
print(f"Davies-Bouldin Score:    {davies_bouldin:.4f}  (lower is better)")
print(f"SSE (Inertia):           {best_sse:.2f}  (lower is better)")
print("="*70)

## 10. Interpretation and Explanation

In [ ]:
# Generate interpretation based on silhouette score
if best_silhouette > 0.5:
    separability = "excellent"
    quality = "strong, well-separated clusters"
elif best_silhouette > 0.3:
    separability = "good"
    quality = "reasonably distinct clusters with some overlap"
elif best_silhouette > 0.2:
    separability = "moderate"
    quality = "moderate cluster structure with considerable overlap"
else:
    separability = "weak"
    quality = "weak cluster structure, potentially artificial groupings"

print("\n" + "="*70)
print("CLUSTER INTERPRETATION")
print("="*70)

explanation = f"""
The optimal number of clusters is K={best_k}, selected based on the highest silhouette 
score of {best_silhouette:.4f}, which indicates {separability} cluster separability. The elbow 
curve analysis supports this choice, showing a notable reduction in SSE from K=2 to K={best_k} 
(SSE={best_sse:.2f}), with diminishing returns beyond this point. The PCA visualization 
reveals {quality}, with the first two principal components explaining 
{pca.explained_variance_ratio_.sum():.1%} of the total variance. The cluster distribution 
shows {'balanced' if max(counts)/min(counts) < 2 else 'somewhat imbalanced'} membership across 
the {best_k} groups, suggesting that the data naturally partitions into {best_k} meaningful 
segments based on the underlying feature patterns.
"""

print(explanation.strip())
print("\n" + "="*70)

## 11. Return Results Summary

In [ ]:
# Create comprehensive results dictionary
results_summary = {
    'best_k': int(best_k),
    'best_silhouette_score': float(best_silhouette),
    'best_sse': float(best_sse),
    'scores_table': scores_df,
    'cluster_distribution': dict(zip([f'Cluster_{i}' for i in unique], counts.tolist())),
    'pca_variance_explained': {
        'PC1': float(pca.explained_variance_ratio_[0]),
        'PC2': float(pca.explained_variance_ratio_[1]),
        'Total': float(pca.explained_variance_ratio_.sum())
    },
    'additional_metrics': {
        'calinski_harabasz_score': float(calinski),
        'davies_bouldin_score': float(davies_bouldin)
    },
    'plots_generated': [
        'kmeans_scores.png',
        'kmeans_pca_clusters.png'
    ],
    'explanation': explanation.strip()
}

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print(f"\n🏆 BEST K: {best_k}")
print(f"\n📊 SCORES TABLE:")
print(scores_df.to_string(index=False))

print(f"\n📈 PCA VARIANCE EXPLAINED:")
print(f"   PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"   PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"   Total: {pca.explained_variance_ratio_.sum():.2%}")

print(f"\n📁 GENERATED PLOTS:")
for plot in results_summary['plots_generated']:
    print(f"   ✓ {plot}")

print(f"\n💡 EXPLANATION:")
print(explanation.strip())

print("\n" + "="*70)
print("✅ KMeans clustering analysis completed successfully!")
print("="*70)

# Return results
results_summary

## Summary

This notebook has successfully:

1. ✅ **Loaded data** and ignored class labels for unsupervised learning
2. ✅ **Scaled features** using StandardScaler fitted on the full X
3. ✅ **Ran KMeans** for K in [2, 3, 4, 5, 6] with n_init='auto' and random_state=42
4. ✅ **Recorded metrics** for each K:
   - Silhouette scores
   - SSE (inertia)
5. ✅ **Selected best K** using highest silhouette score
6. ✅ **Visualized clusters** using PCA (2 components) scatter plot
7. ✅ **Provided interpretation** with 3-4 sentences on:
   - Chosen K justification
   - Cluster separability analysis
   - PCA variance explanation
   - Cluster distribution balance

### Returns:

- **Best K**: Optimal number of clusters
- **Scores Table**: Complete comparison of all K values
- **PCA Plot**: Visual representation of clusters in 2D space
- **Short Explanation**: Detailed interpretation of results

### Additional Features:

- Elbow curve visualization for SSE analysis
- Calinski-Harabasz and Davies-Bouldin scores for additional validation
- Cluster distribution statistics
- Centroid visualization in PCA space